# XAG329A Self-Improving AI Agents - Part 3 Final Assignment

This assignment builds an end-to-end agentic workflow by combining LLMs with external APIs and self-improvement techniques.

- **Part 1:** Implementing an API-augmented LLM pipeline which integrates four external APIs, and generates responses grounded in API responses.
- **Part 2:** Applying self-improvement methods:
  - Query decomposition & fusion: Breaks complex queries into sub-queries, retrieves API outputs, and fuses the results.
  - Iterative refinement: Uses the LLM to trigger additional API calls and refine the response until sufficient information is gathered to answer the query.
- **Part 3:** Integrating these components into a full agentic workflow and evaluating accuracy on the test set.
- **Part 4:** Extending the system into a deep research agent for knowledge-intensive tasks, inspired by recent AI launches.

**Final Deliverable**: Submit `assignment3.ipynb` to Gradescope. Do not rename the file, and run the evaluation cells before submitting so the saved outputs are included. See `README.md` for details.

In [1]:
# Auto-reloads imported modules, so changes to .py files are automatically reflected in the notebook
%load_ext autoreload
%autoreload 2

## Importing the relevant tools.
## Make sure you installed this package with the instructions in README.md
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env at the repo root (walks up from the notebook cwd).
# Refer README.md for more details. Copy `.env.example` to `.env` and add your tokens.
load_dotenv(find_dotenv())


## Package Installation

#### API Configuration and Setup

Before we can use the APIs, we need to set up our environment and initialize the API manager. This involves:

1. Loading API keys from environment variables
2. Setting up the API manager with the necessary credentials
3. Validating that all required keys are present

For this assignment, we'll use:
- Tavily Search API (requires API key)
     - https://www.tavily.com/
     - API description and test notebook: [tavily.ipynb](./eval-search-apis/tavily.ipynb)
- Polygon (Massive) API is a real-time and historical financial market data API (requires API key)
     - https://polygon.io/dashboard/keys
     - rebranded to massive.com
     - API description and test notebook: [polygon.ipynb](./eval-search-apis/polygon.ipynb)

- Wolfram Alpha API (requires **app ID**): 
     - https://developer.wolframalpha.com/access
     - API description and test notebook: [wolframalpha.ipynb](./eval-search-apis/wolframalpha.ipynb)

**Note**: Be mindful of your API usage and compute budget. Use smaller, more cost-effective models (e.g. `openrouter/google/gemma-3-4b-it`) for development before scaling to larger models. For security, never hardcode API keys; always use a `.env` file and add it to your `.gitignore`. See `README.md` for OpenRouter and tool API key setup.

### Model API config

We will use open-source language models through [OpenRouter](https://openrouter.ai/). See `README.md` for how to get API keys and add them to your `.env` file.

OpenRouter provides a single API to many open-source models hosted by different providers, so you do not need to set up a dedicated server. Model availability can change over time; if you cannot access a model used in this notebook, please reach out to the teaching staff.

In [2]:
from xag329a_a3.api_manager import APIManager

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError(
        "OPENROUTER_API_KEY is not set. "
        "Copy .env.example to .env, add your OpenRouter API key "
        "(see README.md), then re-run the imports cell so load_dotenv() runs."
    )
print("OPENROUTER_API_KEY is set.")

TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
POLYGON_API_KEY = os.getenv('POLYGON_API_KEY')
WOLFRAM_APP_ID = os.getenv('WOLFRAM_APP_ID')

if not all([TAVILY_API_KEY, POLYGON_API_KEY, WOLFRAM_APP_ID]):
    raise ValueError(
        "One or more required API keys are missing from environment variables. "
        "Copy .env.example to .env, add TAVILY_API_KEY, POLYGON_API_KEY, and WOLFRAM_APP_ID "
        "(see README.md), then re-run the imports cell so load_dotenv() runs."
    )

api_manager = APIManager(
    tavily_api_key=TAVILY_API_KEY,
    polygon_api_key=POLYGON_API_KEY,
    wolfram_app_id=WOLFRAM_APP_ID
)

print("APIManager initialized successfully.")

### Model config

In this assignment, we will use the following open-source models via OpenRouter.

Judge / cheap eval: `google/gemma-3-4b-it`

- Model page: https://openrouter.ai/google/gemma-3-4b-it

Default / routing: `meta-llama/llama-3.1-8b-instruct`

- Model page: https://openrouter.ai/meta-llama/llama-3.1-8b-instruct

Stronger single-call / decomposition / refinement: `meta-llama/llama-3.3-70b-instruct`

- Model page: https://openrouter.ai/meta-llama/llama-3.3-70b-instruct

Fusion: `meta-llama/llama-3.1-8b-instruct` and `openai/gpt-oss-20b`

- Model pages: https://openrouter.ai/meta-llama/llama-3.1-8b-instruct and https://openrouter.ai/openai/gpt-oss-20b

In [3]:
JUDGE_MODEL = "openrouter/openai/gpt-oss-20b"
DEFAULT_MODEL = "openrouter/meta-llama/llama-3.1-8b-instruct"
STRONG_MODEL = "openrouter/meta-llama/llama-3.3-70b-instruct"
FUSION_MODEL = "openrouter/meta-llama/llama-3.1-8b-instruct"
FUSION_SAMPLE_MODELS = [
    FUSION_MODEL,
    "openrouter/openai/gpt-oss-20b",
]

print(f"Judge model: {JUDGE_MODEL}")
print(f"Default model: {DEFAULT_MODEL}")
print(f"Strong model: {STRONG_MODEL}")
print(f"Fusion model: {FUSION_MODEL}")


## Part 0 - LLM Performance with a Single Call [5 points]

To establish a baseline, let's test the performance of a standalone Large Language Model (LLM) without any external tools or APIs. This helps illustrate the limitations of relying solely on the model's internal knowledge, especially for questions requiring real-time, specific, or computational information.

**Deliverable:**
- In the [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py) file, implement the `generate` method in the MultiLMAgent class.

**Observe that:**
- LLMs can hallucinate or provide outdated information for fact-based queries.
- Models often struggle with precise mathematical calculations that a tool like Wolfram Alpha can solve instantly, due to the tokenization scheme.
- Queries about current stock prices or today's weather are outside the scope of a model's static training data.
- Evaluation is difficult for unstructured model outputs. Numerical answers have multiple possible formats, which makes naive string-matching unreliable; and natural language answers can be phrased in various ways. To that end, we've provided a `evaluate_qa` function that uses a fast, cheap model ([`gemma-3-4b-it`](https://openrouter.ai/google/gemma-3-4b-it)) to more reliably evaluate the model's output.

You can set `DEBUG_MODE` to be `True` when developing. Change to `False` for final results.

In [4]:
DEBUG_MODE = True  # Set to True when developing, False for final results

In [5]:
from xag329a_a3.multi_lm_agent import MultiLMAgent
from xag329a_a3.evaluation import prepare_dataset, evaluate_qa

In [ ]:
# Self-checks for `__init__` and `generate` (no API calls).
# These use a fake API manager and canned replies, so they cost nothing to run.
from xag329a_a3.tests import test_agent_init, test_generate

test_agent_init()
test_generate()

In [6]:
multi_lm_agent = MultiLMAgent(api_manager)
dataset = prepare_dataset(debug_mode=DEBUG_MODE)

queries, answers = dataset['query'], dataset['answer']
num_correct = 0
print("Zero-Shot Responses:")
zero_shot_responses = []
for query, answer in zip(queries, answers):
    response = multi_lm_agent.generate(query=query, model=DEFAULT_MODEL)
    zero_shot_responses.append(response)
    print(f"Query: {query}")
    print(f"Zero-Shot Response: {response}")
    print(f"Answer: {answer}")
    is_correct = evaluate_qa(query, response, answer, model=JUDGE_MODEL)
    num_correct += is_correct
    print("Is Correct:", is_correct)
    print("-" * 50)

accuracy_zero_shot = num_correct / len(queries)
print(f"Accuracy = {accuracy_zero_shot*100:.1f}%")

## Part 1 - API-augmented LLM pipeline [15 points]

In this part, we augment the LLM with four different API calls, then implement an LLM-based API router that routes the query to the appropriate API, and then prompt the LLM to generate the final response given the API outputs.

We will work with the following APIs and select the appropriate API for a given query:

1. **Tavily API** - For web search
2. **Polygon (Massive) API** - For financial data
3. **Wolfram Alpha API** - For mathematical calculations
4. **Weather API** - For location-based weather data

#### 1a. Tavily search API

The Tavily search API allows us to programmatically search the web. We'll use this to gather information and context for our tasks.

Key features:
- Web search with customizable parameters
- Filtering and sorting options
- Rich metadata about search results

Provided: In the [`xag329a_a3/api_manager.py`](xag329a_a3/api_manager.py) file, we provided the ``tavily_search`` functions.

**Note**: The `tavily_search` function will return long webpages, so we will need to truncate or parse the response to get the relevant information. Otherwise, the added context will exceed the context window of the LMs in later functions.

In [7]:
search_query = "Apple Product News"
results = api_manager.tavily_search(
    search_query=search_query,
    num_results=3
)

print("\nSearch results for:", search_query)
print("-" * 50)
for i, result in enumerate(results, 1):
    print(f"Result {i}:")
    for k, v in result.items():
        print(f"\t{k}: {v}")
    print("-" * 50)

#### 1b. Polygon (Massive) API

The Polygon (Massive) API provides real-time and historical financial data. We'll use this for analyzing stock market information.

Key features:
- Real-time stock quotes
- Historical price data
- Technical indicators
- Company fundamentals

Provided: In the [`xag329a_a3/api_manager.py`](xag329a_a3/api_manager.py) file, we provided the ``get_stock_data`` function.

In [8]:
ticker, date = "TSLA", "2025-08-27"
stock_data = api_manager.get_stock_data(ticker=ticker, date=date)
if isinstance(stock_data, dict):
    print(f"\nStock data for {ticker} on {date}:")
    for k, v in stock_data.items():
        print(f"\t{k}: {v}")
else:
    print("\tNo stock data available")

#### 1c. Wolfram Alpha API

The Wolfram Alpha API provides powerful computational capabilities to answer mathematical queries.

Provided: In the [`xag329a_a3/api_manager.py`](xag329a_a3/api_manager.py) file, we provided the `compute` function.

In [9]:
math_query = "integral cos(x)/sqrt(x) from 0 to 1"
result = api_manager.compute(math_query)
print(f"Wolfram Alpha result for {math_query}: {result}")

#### 1d. Weather API

The Weather API provides historical weather data for any location, including temperature, precipitation, wind, etc.

To impement this, we'll use Nominatim from the [Geopy API](https://geopy.readthedocs.io/en/stable/) to geocode a location, and then use the [Open-Meteo API](https://open-meteo.com/) (no API key required) to get the weather data for that location.

Provided: In the [`xag329a_a3/api_manager.py`](xag329a_a3/api_manager.py) file, we provided the ``get_weather`` function.

- API description and test notebook: [open-meteo.ipynb](./eval-search-apis/open-meteo.ipynb)

In [10]:
# We can use the API manager's weather functionality
from datetime import datetime
location = "Palo Alto, CA"
date = datetime.now().strftime('%Y-%m-%d')
hour = "14"

weather_data = api_manager.get_weather(location, date, hour)
if "error" not in weather_data:
    print(f"Weather conditions for {location} on {date} at {hour}:00:")
    for k, v in weather_data.items():
        print(f"\t{k}: {v}")
else:
    print(f"\tCould not get weather data for {location}: {weather_data['error']}")

#### 1e. API Routing [10 points]

The API routing is a system that uses language models to route queries to the appropriate API function. This allows us to build an agent that can use multiple APIs to answer user queries.

Key requirements for the function logic and prompt construction:
- Query an LLM to determine the appropriate API to use for the query
- Correctly parse the query response and map it to the appropriate API function
- In the query response, include the API name and parameters to be used
- Query the selected API and return the response from the API after parsing
- Handle edge cases and fallbacks for query parsing and API selection

**Deliverable**: In the [`xag329a_a3/api_manager.py`](xag329a_a3/api_manager.py) file, implement the ``_parse_query_params`` and ``route_query`` functions.

Note: OpenRouter supports structured outputs, which can simplify the implementation of these functions: https://openrouter.ai/docs/guides/features/structured-outputs.
We use Pydantic models to define the expected parameters for each API.

In [ ]:
# Self-check for `_parse_query_params` and `route_query` (no API calls).
from xag329a_a3.tests import test_api_routing

test_api_routing()

In [11]:
from xag329a_a3.evaluation import prepare_dataset

dataset = prepare_dataset(debug_mode=DEBUG_MODE)
queries, answers = dataset['query'], dataset['answer']
for query in queries:
    print(f"Query: {query}")
    output = api_manager.route_query(query, model=DEFAULT_MODEL)
    print(f"Called `{output['api_used']}` with parameters: {output['params']}")
    print("Result:", output["results"])
    print("API Used:", output["api_used"])
    print("-" * 50)
    break

#### 1f. Evaluation with a single API call [5 points]

With the API router complete, let's evaluate its performance. This involves routing a query to the correct API, getting the data, and then prompting an LLM with both the original query and the API data to generate a final answer.

**Deliverable:**
- In [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py), implement the `single_LM_with_single_API_call` function.`

Key requirements for the function prompt and logic:
- Query the API manager for the necessary data
- Use the query and the data retrieved from the API manager to create a prompt for the model
- Use the model to generate the response
- Return the response from the model

Deliverable: In the [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py) file, implement the `single_LM_with_single_API_call` function.


In [ ]:
# Self-check for `single_LM_with_single_API_call` (no API calls) -- run before the evaluation below.
from xag329a_a3.tests import test_single_lm_with_single_api_call

test_single_lm_with_single_api_call()

In [12]:
from xag329a_a3.evaluation import evaluate_qa, prepare_dataset

multi_lm_agent = MultiLMAgent(api_manager)
dataset = prepare_dataset(debug_mode=DEBUG_MODE)

queries, answers = dataset['query'], dataset['answer']
num_correct = 0

print("Single-Call LM with Single API Call:")
single_LM_with_single_API_call_responses = []
for query, answer in zip(queries, answers):
    response = multi_lm_agent.single_LM_with_single_API_call(query=query, model=STRONG_MODEL)
    single_LM_with_single_API_call_responses.append(response)
    print(f"Query: {query}")
    print(f"Single-Call LM Response: {response}")
    print(f"Answer: {answer}")
    is_correct = evaluate_qa(query, response, answer, model=JUDGE_MODEL)
    num_correct += is_correct
    print("Is Correct:", is_correct)
    print("-" * 50)

accuracy_singleLM_with_single_API_call = num_correct / len(queries)
print(f"Accuracy = {accuracy_singleLM_with_single_API_call*100:.1f}%")

## Part 2 - Self-improvement techniques [20 points]
We now improve the accuracy by using a) **query decomposition and fusion** and b) **iterative self-refinement**

#### 2a. Query Decomposition [5 points]

Complex queries often require information from multiple API calls (e.g., "Which city was windier, Chicago or Boston?"). To address such queries, we will first build the Query Decomposition component, which breaks down complex queries into simpler, more manageable parts, allowing us to use multiple APIs to answer the query.

Key requirements:
- Write the decomposition prompt: instruct the LLM to break the query into independent sub-queries, each wrapped in `<sub-query>...</sub-query>` tags (`_get_query_decomposition_prompt`)
- Send that prompt to the decomposition model (`_get_sub_queries`); parsing the tags and capping the count at `max_sub_queries` is already provided

**Deliverable**: In the [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py) file, implement the `_get_query_decomposition_prompt` and `_get_sub_queries` helper functions.

**Note**:
- `decompose_query` is provided for you. It routes each sub-query through the API manager and gathers the results, using a `ThreadPoolExecutor` to run those calls concurrently, which can significantly speed up execution.
- `decompose_query` returns one dictionary per sub-query, holding the sub-query, the API used, the parameters, the results, and any error. Part 2b consumes these dictionaries.
- Tavily results carry both a truncated `snippet` and the full `webpage_content`. Prefer the snippet when feeding results into a prompt, to avoid exceeding the context window of the LMs.
- How do these sub-queries help with the overall task? What information do they provide that the original query does not?

In [ ]:
# Self-check for the decomposition prompt and `_get_sub_queries` (no API calls).
from xag329a_a3.tests import test_query_decomposition

test_query_decomposition()

In [13]:
from xag329a_a3.multi_lm_agent import MultiLMAgent
from xag329a_a3.evaluation import prepare_dataset, evaluate_qa

multi_lm_agent = MultiLMAgent(api_manager)

dataset = prepare_dataset(debug_mode=DEBUG_MODE)
queries, answers = dataset['query'], dataset['answer']

print("Testing Query Decomposition:")
decomposed_queries = []  # List[List[Dict]], where each inner list contains the decomposed queries for a single query
for query in queries:
    print(f"Original Query: {query}")
    decomposed_query = multi_lm_agent.decompose_query(query=query, max_sub_queries=3)

    print("Sub-queries:")
    for sub_query in decomposed_query:
        for k, v in sub_query.items():
            print(f"\t{k}: {v}")
        print()
    print("-" * 50)

    decomposed_queries.append(decomposed_query)

#### 2b. Synthesizing information from sub-query API responses [5 points]

After executing the sub-queries, we need to assemble a new, context-rich prompt that incorporates each sub-query's API response and the user's query. This will be given to a model to synthesize a final answer.

**Deliverable**: In the [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py) file, implement `_get_synthesis_prompt`.

**Note**:
- Do the prompts constructed provide all the necessary information to answer the query?
- A good synthesis prompt should clearly present the original query and  neatly format the results from each successful API call

In [ ]:
# Self-check for `_get_synthesis_prompt` (no API calls).
from xag329a_a3.tests import test_synthesis_prompt

test_synthesis_prompt()

In [14]:
dataset = prepare_dataset(debug_mode=DEBUG_MODE)

queries, answers = dataset['query'], dataset['answer']
generated_prompts = []
for query, decomposed_query in zip(queries, decomposed_queries):
    generated_prompt = multi_lm_agent._get_synthesis_prompt(query, decomposed_query)
    print("Generated Prompt:")
    print(generated_prompt)
    print("-" * 50)
    generated_prompts.append(generated_prompt)

#### 2c. Fusion of responses [5 points]

With the constructed prompt, we can generate multiple responses with different models to get a diverse set of responses, and then use a fusion model to combine the best elements from each into a single coherent output. This generally provides a more robust, comprehensive response than any single model could provide.

Key requirements:
- Call `decompose_query` and `_get_synthesis_prompt` to gather the sub-query results and turn them into the synthesis prompt (which contains the original query and the results from each sub-query)
- Send that one synthesis prompt to each of the sampling models in `self.fusion_sample_models`, collecting their answers (these default to the same two models as the notebook's `FUSION_SAMPLE_MODELS`: `"openrouter/meta-llama/llama-3.1-8b-instruct"` and `"openrouter/openai/gpt-oss-20b"`)
- Combine those answers by writing a new fusion prompt that presents them alongside the original query, and send it to `self.fusion_model`
- Handle edge cases and fallbacks: `generate` returns `None` when a model fails, so fall back to what you do have rather than losing the query
- Return the final response from the fusion model

**Deliverable**: In the [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py) file, implement the `decompose_and_fuse` method. It orchestrates the full query decomposition and fusion workflow, calling `decompose_query`, `_get_synthesis_prompt`, and `generate` internally.

**Note**:
- The diversity that makes fusion worthwhile comes from asking different models the same question, rather than from varying the prompt, so every sampling model receives the identical synthesis prompt.
- Compare how the fused response differs from the individual sampled responses. Does fusing actually improve the answer, or does it blur the best response into the others?

In [ ]:
# Self-check for `decompose_and_fuse` (no API calls).
from xag329a_a3.tests import test_decompose_and_fuse

test_decompose_and_fuse()

In [15]:
from xag329a_a3.evaluation import evaluate_qa, prepare_dataset

dataset = prepare_dataset(debug_mode=DEBUG_MODE)
queries, answers = dataset['query'], dataset['answer']

print("Generated Responses with Query Decomposition + Fusion:")
query_decomp_responses = []
num_correct = 0
for query, answer in zip(queries, answers):
    response = multi_lm_agent.decompose_and_fuse(query)
    query_decomp_responses.append(response)
    print(f"Query: {query}")
    print(f"Query Decomposition + Fusion Response: {response}")
    print(f"Answer: {answer}")
    is_correct = evaluate_qa(query, response, answer, model=JUDGE_MODEL)
    num_correct += is_correct
    print("Is Correct:", is_correct)
    print("-" * 50)

accuracy_query_decomp = num_correct / len(queries)
print(f"Accuracy = {accuracy_query_decomp*100:.1f}%")

#### 2d. Iterative Self-Refinement [5 points]

Sometimes a single round of decomposition is not enough to answer the query. To address this, Iterative Refinement improves the response by querying for more information as needed across multiple APIs.

At each step, the agent can either issue a new sub-query to gather more information (e.g. if a prior sub-query failed or if additional information is needed), or generate the final answer if it has enough information. This can be particularly useful for complex queries that require multiple API calls to answer, such as multi-hop question-answering where the result of one step informs the next.

Key requirements:
- Build the prompt for the current step by calling `_get_iterative_refinement_prompt` with the original user query and the history gathered so far
- Send that prompt to the iterative refinement model, keeping the reply as raw text; the loop below parses the tags

**Deliverable**: In [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py), implement the marked block inside the loop of the `iterative_refine` method.

**Note**:
- `_get_iterative_refinement_prompt` is provided for you. It formats the history of earlier sub-queries and instructs the model to reply with either `<sub-query>...</sub-query>` or `<final_answer>...</final_answer>`.
- The surrounding loop is provided as well: it reads the model's choice, sends any new sub-query through `route_query`, records the outcome in the history, and stops once the model gives a final answer or the iteration budget runs out.
- Printing the history on each pass is worth the few extra lines, since it is your only view into why a multi-hop answer went wrong.
- Unlike query decomposition + fusion, iterative refinement can re-issue a sub-query when an earlier one failed, and can feed intermediate results into later sub-queries. Which kinds of queries does each approach handle better?

For resources on LM judges and self-verification, see: 
- [LM Judge Survey](https://arxiv.org/abs/2411.15594)
- [Large Language Models are Better Reasoners with Self-Verification](https://arxiv.org/abs/2212.09561)

In [ ]:
# Self-check for `iterative_refine` (no API calls).
from xag329a_a3.tests import test_iterative_refine

test_iterative_refine()

In [16]:
from xag329a_a3.evaluation import evaluate_qa, prepare_dataset
from tqdm import tqdm

dataset = prepare_dataset(debug_mode=DEBUG_MODE)
queries, answers = dataset['query'], dataset['answer']

multi_lm_agent = MultiLMAgent(api_manager)
print("Generated Responses with Iterative Refinement:")
iterative_refine_responses = []
num_correct = 0
for query, answer in tqdm(zip(queries, answers)):
    print(f"Query: {query}")
    
    response = multi_lm_agent.iterative_refine(query, max_iterations=4)
    print(f"Iterative Refinement Response: {response}")
    iterative_refine_responses.append(response)
    
    print(f"Answer: {answer}")
    is_correct = evaluate_qa(query, response, answer, model=JUDGE_MODEL)
    num_correct += is_correct
    print("Is Correct:", is_correct)
    print("-" * 50)
    
accuracy_iterative_refine = num_correct / len(queries)
print(f"Accuracy = {accuracy_iterative_refine*100:.1f}%")

Create a bar graph plotting the accuracies of zero-shot prompting, a single API router and LM call, query decomposition and fusion, and iterative refinement (max 4 rounds).

In [21]:
from matplotlib import pyplot as plt

# Create data for the bar graph
methods = ['Zero-shot', 'Single API router and LM call', 'Query decomposition and fusion', 'Iterative refinement']

# Must be in this order and in floating point format from 0.0 to 1.0
accuracies = [
    accuracy_zero_shot,
    accuracy_singleLM_with_single_API_call,
    accuracy_query_decomp,
    accuracy_iterative_refine
]

colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, accuracies, color=colors)

plt.title('Accuracy Comparison Across Different LLM Pipelines')
plt.ylabel('Accuracy')
plt.ylim(0, 1.0)

plt.xticks(rotation=45, ha='right')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2%}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## Part 3 - Building an LLM Agentic Workflow [10 points]

Using the components we've built (or new ones you've implemented), build an LLM agent that can answer the following questions about the world. These questions will be a mix of the types of questions we've built components for and will require using the APIs in creative ways.

Some questions will require just a single API call, while others will require multiple API calls and multiple rounds of iterative refinement. Create a pipeline that can dynamically adjust to the complexity of the question. Feel free to implement new components or use the ones we've already built!

If you get above 55% accuracy on the entire dataset, you will get full points. For scores below 55%, you will get partial credit based on the percentage of accuracy.

**Important**: Make sure to evaluate over the entire dataset when you are confident with your implementation! This will help you preserve inference compute credits and speed up the development process. Please use `STRONG_MODEL` (`openrouter/meta-llama/llama-3.3-70b-instruct`) for the iterative refinement and decomposition models, and `FUSION_MODEL` (`openrouter/meta-llama/llama-3.1-8b-instruct`) for the fusion model.

Deliverable: In the [`xag329a_a3/multi_lm_agent.py`](xag329a_a3/multi_lm_agent.py) file, implement the `run_pipeline` method, which takes a query and returns a final response.

**Note**: How does the performance on the dataset compare between single-call LMs vs. the complete pipeline with the multi-LM agent? How does it improve accuracy by improving access to tool APIs and allowing for more complex reasoning?

Let's first evaluate over the entire dataset with just a zero-shot model
- **IMPORTANT**: Make sure to evaluate over the entire dataset only when you are confident with your implementation! 
- This will help you preserve inference compute credits and speed up the development process.

In [ ]:
# Self-check for `run_pipeline` (no API calls) -- run this before the full evaluation below.
from xag329a_a3.tests import test_run_pipeline

test_run_pipeline()

In [18]:
# make sure to evaluate over the entire dataset only when you are confident with your implementation! 
DEBUG_MODE = False # Loads the entire dataset for evaluation.

In [19]:
# GRADED CELL -- the autograder reads the "Accuracy = ..." line below out of your
# saved notebook and uses it as the baseline your agentic pipeline has to beat.
# Set DEBUG_MODE = False above, run this cell, then save the notebook.
# Let's first evaluate the zero-shot performance over the entire dataset
# IMPORTANT: Make sure to evaluate over the entire dataset when you are confident with your implementation! 
# This will help you preserve inference compute credits and speed up the development process.

from xag329a_a3.evaluation import evaluate_qa, prepare_dataset

dataset = prepare_dataset(debug_mode=DEBUG_MODE)
queries, answers = dataset['query'], dataset['answer']

zero_shot_responses, num_correct = [], 0
for query, answer in zip(queries, answers):
    print(f"Query: {query}")
    response = multi_lm_agent.generate(query, model=STRONG_MODEL)
    zero_shot_responses.append(response)
    print(f"Zero-Shot Response: {response}")
    print(f"Answer: {answer}")
    is_correct = evaluate_qa(query, response, answer, model=JUDGE_MODEL)
    num_correct += is_correct
    print("Is Correct:", is_correct)
    print("-" * 50)

print("Evaluating full zero-shot performance...")
complete_set_zero_shot_accuracy = num_correct / len(queries)
print(f"Accuracy = {complete_set_zero_shot_accuracy*100:.1f}%")


Now let's evaluate over the entire dataset with our multi-LM agentic pipeline

In [20]:
# GRADED CELL -- the autograder reads the "Accuracy = ..." line below out of your
# saved notebook. Set DEBUG_MODE = False above, run this cell, then save the
# notebook. Keep the per-query prints: they show how many questions were graded.
from xag329a_a3.evaluation import evaluate_qa, prepare_dataset

dataset = prepare_dataset(debug_mode=DEBUG_MODE)
queries, answers = dataset['query'], dataset['answer']

multi_lm_agent = MultiLMAgent(api_manager)

multi_lm_responses, num_correct = [], 0
print("\nGenerating responses with Multi-LM Agent...")
for query, answer in zip(queries, answers):
    print(f"Query: {query}")
    response = multi_lm_agent.run_pipeline(query)
    multi_lm_responses.append(response)
    print(f"Multi-LM Response: {response}")
    print(f"Answer: {answer}")
    is_correct = evaluate_qa(query, response, answer, model=JUDGE_MODEL)
    num_correct += is_correct
    print("Is Correct:", is_correct)
    print("-" * 50)

complete_set_multi_lm_agent_accuracy = num_correct / len(queries)
print(f"Accuracy = {complete_set_multi_lm_agent_accuracy*100:.1f}%")


## Part 4 - Deep Research Agent [Optional]

Agentic LM systems are used everywhere today! From chatbots to coding agents to task automation, they are becoming more and more prevalent in our daily lives. Gemini, OpenAI, and Perplexity (among others) have "deep research" agents that are capable of synthesizing large amounts of online information and completing multi-step research tasks: [Introducing Deep Research](https://openai.com/index/introducing-deep-research/), [Deep Research](https://blog.google/products/gemini/google-gemini-deep-research/).

Using the components we've built and extending them if needed, implement your own deep research agent that can generate comprehensive analyses from online sources. The agent should be able to handle complex queries requiring multi-step research, synthesizing information from multiple sources, and generating a comprehensive final report.

**Key requirements for implementation:**
- Generate a four-five paragraph report
- Proper, easy-to-read structuring of the report
- Usage of multiple sources of information with appropriate link citations
- Track temporal information and maintain chronological accuracy

Optional: In the [`xag329a_a3/DeepResearchAgent.py`](xag329a_a3/DeepResearchAgent.py) file, implement the `research` method in the `DeepResearchAgent` class.

The method should:
- Take a complex query (e.g., "What was the UK's macroeconomic performance in 2024?")
- Break it down into sub-questions
- Research each sub-question using the search engine API
- Synthesize and summarize findings with appropriate formatting as a report
- Return a report and list of sources 
- **IMPORTANT**: Make sure to use cheaper models during the development process to help you preserve inference compute credits and speed up the process!

This is an optional part where you can experiment with and evaluate a deep research agent for complex questions and research requests.


In [ ]:
from xag329a_a3.DeepResearchAgent import DeepResearchAgent

test_queries = [
    "What are the key developments and challenges in solid-state battery technology for electric vehicles in 2024, including major company announcements and technical breakthroughs?",
    "How has the implementation of the UK's post-Brexit immigration policy affected its labor market and key industries between 2021-2024? Include specific policy changes and their measured impacts.",
    "What progress has been made in nuclear fusion energy in 2024, focusing on major research milestones, private sector investments, and timeline predictions for commercial viability?"
]

custom_agent = DeepResearchAgent(api_manager)

for query in test_queries:
    print(f"\nQuery: {query}")
    single_LM_response = custom_agent.generate(query, model=DEFAULT_MODEL)
    report = custom_agent.research(query)
    print(f"Single-LM Response: {single_LM_response}")
    print("\n" * 3)
    print(f"Report: {report['report']}")
    print(f"Sources: {report['sources']}")
    print("-" * 50)